In [117]:
import h3
import pandas as pd
import folium
import branca.colormap as cm

In [118]:
import geopandas as gpd

In [2]:
pip install h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.9 MB/s eta 0:00:00


# Test h3 with 311 data

In [119]:
Stops = gpd.read_file('/content/AllStops_LatLong.csv')

In [56]:
import shapely

In [120]:
Stops = (gpd.read_file('/content/AllStops_LatLong.csv')
       .assign(geometry=lambda _df: shapely.wkt.loads(_df['geometry']))
       .set_geometry('geometry').set_crs(2039))

In [13]:
from shapely import wkt

In [121]:
def get_lat(x):
  return x.coords.xy[1][0]

def get_long(x):
  return x.coords.xy[0][0]

In [44]:
type(Stops['geometry'][0])

str

In [122]:
Stops['X'] = Stops['geometry'].apply(lambda x: get_long(x))
Stops['Y'] = Stops['geometry'].apply(lambda x: get_lat(x))

In [123]:
Stops

,field_1,ID,X,Y,Area,geometry
0,0,36460,32.807061,35.128169,Haifa,POINT (32.807 35.128)
1,1,36748,32.767511,35.011954,Haifa,POINT (32.768 35.012)
2,2,32005,32.773228,35.019801,Haifa,POINT (32.773 35.02)
3,3,32006,32.763179,35.017356,Haifa,POINT (32.763 35.017)
4,4,36090,32.767863,34.969340,Haifa,POINT (32.768 34.969)
...,...,...,...,...,...,...
1283,1283,512232,31.889472,34.781162,Ashdod-Ashkelon,POINT (31.889 34.781)
1284,1284,512233,31.891040,34.784951,Ashdod-Ashkelon,POINT (31.891 34.785)
1285,1285,512117,31.892880,34.790228,Ashdod-Ashkelon,POINT (31.893 34.79)
1286,1286,512235,31.903580,34.792379,Ashdod-Ashkelon,POINT (31.904 34.792)


# Create function to parse each row to create H3 index string and geometry boundaries

In [124]:
def compute_h3_and_boudaries(row, resolution=9):
  lat = row['X']
  lng = row['Y']
  h3_index = h3.latlng_to_cell(lat, lng, resolution)
  boundary = h3.cell_to_boundary(h3_index)
  return pd.Series([h3_index, boundary])

# Implement function and check the results

In [125]:
Stops[['H3_Index', 'Boundary']] = Stops.apply(compute_h3_and_boudaries, axis=1)
Stops[['H3_Index', 'Boundary']]

,H3_Index,Boundary
0,892db0aec1bffff,"((32.80743764597494, 35.12445533881866), (32.8..."
1,892db0a1d97ffff,"((32.769160038158454, 35.01038375702672), (32...."
2,892db0a1db7ffff,"((32.7739237598238, 35.01914468435645), (32.77..."
3,892db0a1d8fffff,"((32.765518319361, 35.01578566566435), (32.763..."
4,892db0a0003ffff,"((32.7689225592109, 34.96640068702931), (32.76..."
...,...,...
1283,892db053023ffff,"((31.89022699412384, 34.77833002011167), (31.8..."
1284,892db05315bffff,"((31.892389166359074, 34.78461482642633), (31...."
1285,892db053143ffff,"((31.895039937465366, 34.78701474982541), (31...."
1286,892db05310fffff,"((31.90348085877399, 34.79032991143254), (31.9..."


# Aggregate by H3 index to get count of reports

In [126]:
Stops_counts = Stops.groupby('H3_Index').size().reset_index(name='Count')
Stops_counts

,H3_Index,Count
0,892db002917ffff,2
1,892db002b97ffff,1
2,892db00348fffff,1
3,892db006147ffff,1
4,892db00615bffff,1
...,...,...
1022,892db62e607ffff,1
1023,892db62e68fffff,1
1024,892db62f32fffff,1
1025,892db62f337ffff,1


In [89]:
Stops_counts.sort_values(by='Count', ascending=False)

,H3_Index,Count
11152,8a3f4dc1168ffff,12
6214,8a3f4d848707fff,12
6416,8a3f4d84b1b7fff,10
6391,8a3f4d84b08ffff,10
7074,8a3f4d86bb8ffff,9
...,...,...
7090,8a3f4d86d02ffff,1
7091,8a3f4d86d067fff,1
7092,8a3f4d86d08ffff,1
7093,8a3f4d86d0c7fff,1


# Merge in boundaries to the counts

In [127]:
Stops_counts = Stops_counts.merge(Stops[['H3_Index', 'Boundary']].drop_duplicates(), on='H3_Index', how='left')
Stops_counts

,H3_Index,Count,Boundary
0,892db002917ffff,2,"((32.41614624056273, 35.00519018382992), (32.4..."
1,892db002b97ffff,1,"((32.39831258788591, 35.00629600623695), (32.3..."
2,892db00348fffff,1,"((32.305732523764114, 34.89993479176875), (32...."
3,892db006147ffff,1,"((32.502547142875365, 35.066923702100794), (32..."
4,892db00615bffff,1,"((32.497275026656624, 35.06208760153684), (32...."
...,...,...,...
1022,892db62e607ffff,1,"((31.648154729469333, 34.70660077430163), (31...."
1023,892db62e68fffff,1,"((31.649122995115057, 34.698845305435015), (31..."
1024,892db62f32fffff,1,"((31.602486006960003, 34.764640822728026), (31..."
1025,892db62f337ffff,1,"((31.60611154935086, 34.75928455607557), (31.6..."


In [128]:
Stops_counts['Count'] = Stops_counts['Count'].astype(int)

In [113]:
Stops_counts[Stops_counts['Count']>0]

,H3_Index,Count,Boundary
0,892db000013ffff,1,"((32.33260773141402, 34.98975130460096), (32.3..."
1,892db000067ffff,1,"((32.330610638240245, 35.005336625031674), (32..."
2,892db00006fffff,1,"((32.32747138490817, 35.00681944381477), (32.3..."
3,892db000073ffff,1,"((32.32847017749163, 34.99902734911003), (32.3..."
4,892db00009bffff,1,"((32.33360562026602, 34.98195791218541), (32.3..."
...,...,...,...
10974,892db635b67ffff,3,"((31.779298431110274, 34.62622510112775), (31...."
10975,892db635b6bffff,5,"((31.773502767368306, 34.6253196547028), (31.7..."
10976,892db635b6fffff,4,"((31.776159038552613, 34.62771464739795), (31...."
10977,892db635b77ffff,1,"((31.779781536396325, 34.62234034959689), (31...."


# Visualize on map

In [129]:
m = folium.Map(location=[31.686625960483177, 34.60435986202363], zoom_start=16, tiles='cartodbpositron')

colormap = cm.LinearColormap(colors=['#ffcccc','#ff0000'], vmin=Stops_counts[Stops_counts['Count']>0]['Count'].min(), vmax=Stops_counts[Stops_counts['Count']>0]['Count'].max())

for _, row in Stops_counts[Stops_counts['Count']>0].iterrows():
  boundaries = row['Boundary']
  count = row['Count']
  color = colormap(count)
  folium.Polygon(
      locations=boundaries,
      color=color,
      weight=1,
      fill=True,
      fill_opacity=0.7,
      popup=f"Count: {count}"
  ).add_to(m)

colormap.add_to(m)

In [130]:
m

In [32]:
Stops_counts

,H3_Index,Count,Boundary
0,8a000030c877fff,1,"((79.5273133212226, 36.827931711246286), (79.5..."
1,8a0000860c37fff,1,"((79.03377137111329, 35.93143373965706), (79.0..."
2,8a000129cd5ffff,1,"((80.53496083806928, 34.45049524236374), (80.5..."
3,8a000156bb4ffff,1,"((80.0457017526631, 35.36145589474203), (80.04..."
4,8a00020094affff,1,"((79.3213641619296, 43.67731898078721), (79.32..."
...,...,...,...
25265,8af3b5372a5ffff,1,"((-78.9108391454888, 167.2595448723149), (-78...."
25266,8af3b63568affff,1,"((-79.08423049534542, 157.94217710451656), (-7..."
25267,8af3b68c1957fff,1,"((-78.6287806388903, 157.344916617162), (-78.6..."
25268,8af3b68f6b1ffff,1,"((-78.56539866484435, 157.0381908358202), (-78..."
